# Etapa 3 — Camada Silver

Pipeline de limpeza e preparação dos dados Bronze para Machine Learning.

**Princípios:**
- Cada dataset é tratado de forma independente (sem joins — isso é função da Gold)
- A `quality_flag` da Etapa 2 orienta decisões por linha durante a limpeza; é descartada ao final
- Nulos são interpretados pelo seu **significado de negócio**, não apenas pelo percentual técnico
- Todas as decisões de descarte e imputação são documentadas automaticamente em `docs/`

**Entradas:** `data/bronze/*_validated.parquet`  
**Saídas:** `data/silver/*_silver.parquet` + `docs/silver_decisions.md` + `docs/anti_leakage_checklist.md`

In [31]:
## Célula 1 — Setup e caminhos
from pathlib import Path
from datetime import datetime, timezone
import pandas as pd
import numpy as np

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

BRONZE_PATH = PROJECT_ROOT / "data" / "bronze"
SILVER_PATH = PROJECT_ROOT / "data" / "silver"
DOCS_PATH   = PROJECT_ROOT / "docs"

for p in (SILVER_PATH, DOCS_PATH):
    p.mkdir(parents=True, exist_ok=True)

print(f"PROJECT_ROOT : {PROJECT_ROOT}")
print(f"BRONZE_PATH  : {BRONZE_PATH}")
print(f"SILVER_PATH  : {SILVER_PATH}")
print(f"DOCS_PATH    : {DOCS_PATH}")

PROJECT_ROOT : C:\Users\dti-\Pessoal\CC5\ciencia-de-dados\cybersecurity-breach-data-project
BRONZE_PATH  : C:\Users\dti-\Pessoal\CC5\ciencia-de-dados\cybersecurity-breach-data-project\data\bronze
SILVER_PATH  : C:\Users\dti-\Pessoal\CC5\ciencia-de-dados\cybersecurity-breach-data-project\data\silver
DOCS_PATH    : C:\Users\dti-\Pessoal\CC5\ciencia-de-dados\cybersecurity-breach-data-project\docs


In [32]:
## Célula 2 — Leitura dos datasets validados

validated_files = {
    "incidents_master":  BRONZE_PATH / "incidents_master_validated.parquet",
    "financial_impact":  BRONZE_PATH / "financial_impact (1)_validated.parquet",
    "market_impact":     BRONZE_PATH / "market_impact_validated.parquet",
}

raw = {}
for name, path in validated_files.items():
    df = pd.read_parquet(path)
    raw[name] = df
    print(f"=== {name} ===")
    print(f"  Shape: {df.shape}")
    print(f"  quality_flag distribuição:")
    print(df["quality_flag"].value_counts(dropna=False).to_string())
    print()

=== incidents_master ===
  Shape: (850, 35)
  quality_flag distribuição:
quality_flag
NULL_ALERT|NULL_CRITICAL               614
NULL_CRITICAL                          232
DATE_RANGE|NULL_CRITICAL                 2
DATE_RANGE|NULL_ALERT|NULL_CRITICAL      2

=== financial_impact ===
  Shape: (778, 22)
  quality_flag distribuição:
quality_flag
NULL_CRITICAL               435
NULL_ALERT|NULL_CRITICAL    336
NULL_ALERT                    7

=== market_impact ===
  Shape: (358, 34)
  quality_flag distribuição:
quality_flag
NULL_CRITICAL               239
OK                           83
NULL_ALERT|NULL_CRITICAL     27
NULL_ALERT                    9



In [33]:
## Célula 3 — Diagnóstico Bronze (resumo por dataset)

for name, df in raw.items():
    total = len(df)
    print("=" * 70)
    print(f"DIAGNÓSTICO: {name}")
    print("=" * 70)
    print(f"  Linhas: {total} | Colunas: {len(df.columns)}")

    # Duplicatas
    meta_cols = ["ingestion_timestamp", "source_file", "quality_flag"]
    data_cols  = [c for c in df.columns if c not in meta_cols]
    dup_exact  = df.duplicated(subset=data_cols, keep=False).sum()
    print(f"  Duplicatas exatas: {dup_exact}")
    if "incident_id" in df.columns:
        dup_key = df.duplicated(subset=["incident_id"], keep=False).sum()
        print(f"  Duplicatas por incident_id: {dup_key}")

    # Nulos por coluna (apenas as que têm nulos)
    null_counts = df.isnull().sum()
    null_counts = null_counts[null_counts > 0].sort_values(ascending=False)
    if not null_counts.empty:
        print(f"\n  Colunas com nulos ({len(null_counts)}):")
        for col, cnt in null_counts.items():
            print(f"    {col:<45} {cnt:>5} ({cnt/total*100:.1f}%)")
    else:
        print("  Sem nulos.")
    print()

DIAGNÓSTICO: incidents_master
  Linhas: 850 | Colunas: 35
  Duplicatas exatas: 0
  Duplicatas por incident_id: 0

  Colunas com nulos (12):
    review_flag                                     780 (91.8%)
    industry_secondary                              697 (82.0%)
    attack_vector_secondary                         639 (75.2%)
    notes                                           636 (74.8%)
    data_source_secondary                           464 (54.6%)
    stock_ticker                                    438 (51.5%)
    downtime_hours                                  430 (50.6%)
    attribution_confidence                          368 (43.3%)
    attributed_group                                368 (43.3%)
    attack_chain                                    275 (32.4%)
    data_type                                       248 (29.2%)
    data_compromised_records                        248 (29.2%)

DIAGNÓSTICO: financial_impact
  Linhas: 778 | Colunas: 22
  Duplicatas exatas: 0
  Duplicat

## Limpeza — `incidents_master`

Decisões baseadas na análise de negócio + relatório de qualidade (ver `PLANO_ETAPA_3.md` para justificativas completas).

**Descartes:** `company_name`, `industry_secondary`, `attack_chain`, `review_flag`, `notes`, `quality_score`, `quality_grade`, `confidence_tier`, `data_source_primary`, `data_source_secondary`, `created_at`, `updated_at`  
**Nulos estruturais (não imputar):** `stock_ticker` (empresas privadas não têm ticker)  
**Imputar `"unknown"`:** `attributed_group`, `attribution_confidence`, `data_type`  
**Imputar mediana + flag:** `data_compromised_records`, `downtime_hours`  
**Converter para datetime:** `incident_date`, `discovery_date`, `disclosure_date`  
**Remover linhas:** datas fora do range [1990, hoje] + incoerência temporal  
**Leakage removido:** `quality_score`, `quality_grade`, `confidence_tier`, `review_flag`, `disclosure_date` (raw — usada só para derivar `days_to_disclosure`)

In [34]:
## Célula 4 — Limpeza: incidents_master

df_inc = raw["incidents_master"].copy()
today = pd.Timestamp.now(tz=None)

# --- 1. Deduplicação ---
before = len(df_inc)
df_inc = df_inc.drop_duplicates(subset="incident_id", keep="first")
print(f"Deduplicação por incident_id: {before - len(df_inc)} removidas")

# --- 2. Flag de vetor secundário (ANTES de descartar a coluna) ---
df_inc["has_secondary_vector"] = df_inc["attack_vector_secondary"].notna().astype(int)

# --- 3. Colunas a descartar ---
# Leakage: quality_score, quality_grade, confidence_tier, review_flag
# Alta cardinalidade / texto livre: company_name, attack_chain, notes, data_source_primary, data_source_secondary
# Nulos estruturais acima de 70% sem valor preditivo: industry_secondary, attack_vector_secondary
# Metadados de sistema com zero variância: created_at, updated_at
DISCARD_INC = [
    "company_name",           # identificador — não é feature preditiva
    "industry_secondary",     # 82% nulos estruturais; vetor primário já presente
    "attack_vector_secondary",# 75% nulos estruturais; já capturado em has_secondary_vector
    "attack_chain",           # texto narrativo livre; não adequado para ML clássico
    "data_source_primary",    # identificador de fonte, não atributo do incidente
    "data_source_secondary",  # idem + 55% nulos
    "quality_score",          # LEAKAGE: score interno calculado após análise do incidente
    "quality_grade",          # LEAKAGE: derivado do quality_score
    "confidence_tier",        # LEAKAGE: metadado de curadoria interna pós-ingestão
    "review_flag",            # LEAKAGE: anotação humana posterior ao incidente
    "notes",                  # texto livre; 75% nulos
    "created_at",             # metadado de sistema; valor idêntico para todos os registros
    "updated_at",             # idem
]
df_inc = df_inc.drop(columns=[c for c in DISCARD_INC if c in df_inc.columns])

# --- 4. Padronização de categorias (trim + lower + strings vazias → None) ---
str_cat_cols = [
    "country_hq", "industry_primary", "attack_vector_primary",
    "attributed_group", "attribution_confidence", "data_type",
    "data_source_type", "systems_affected",
]
for col in str_cat_cols:
    if col in df_inc.columns:
        df_inc[col] = df_inc[col].str.strip().str.lower()
        df_inc[col] = df_inc[col].where(df_inc[col] != "", other=None)

# --- 5. Imputação de nulos ---
# stock_ticker: NÃO imputar — nulo é semanticamente correto para empresas privadas
# attributed_group + attribution_confidence: par condicional — imputar "unknown"
df_inc["attributed_group"]     = df_inc["attributed_group"].fillna("unknown")
df_inc["attribution_confidence"] = df_inc["attribution_confidence"].fillna("unknown")
# data_type: "none" onde não houve exfiltração; "unknown" onde houve mas não foi documentado
if "data_type" in df_inc.columns:
    no_exfil = df_inc["data_compromised_records"].notna() & (df_inc["data_compromised_records"] == 0)
    df_inc.loc[df_inc["data_type"].isna() & no_exfil, "data_type"] = "none"
    df_inc["data_type"] = df_inc["data_type"].fillna("unknown")

# data_compromised_records: imputar mediana + flag (nulo ambíguo: sem exfiltração OU não reportado)
df_inc["data_loss_unknown"] = df_inc["data_compromised_records"].isna().astype(int)
median_dcr = df_inc["data_compromised_records"].median()
df_inc["data_compromised_records"] = df_inc["data_compromised_records"].fillna(median_dcr)

# downtime_hours: imputar mediana + flag (0 ≠ nulo: 0=sem downtime; nulo=não informado)
df_inc["downtime_unknown"] = df_inc["downtime_hours"].isna().astype(int)
median_dh = df_inc["downtime_hours"].median()
df_inc["downtime_hours"] = df_inc["downtime_hours"].fillna(median_dh)

# --- 6. Conversão de datas ---
for col in ["incident_date", "discovery_date", "disclosure_date"]:
    if col in df_inc.columns:
        df_inc[col] = pd.to_datetime(df_inc[col], errors="coerce", utc=False)
        # remover timezone se presente (pandas 3.x pode inferir UTC de alguns formatos)
        if hasattr(df_inc[col].dt, "tz") and df_inc[col].dt.tz is not None:
            df_inc[col] = df_inc[col].dt.tz_localize(None)

# --- 7. Remover linhas com datas fora do range [1990-01-01, hoje] ---
DATE_MIN = pd.Timestamp("1990-01-01")
before = len(df_inc)
for col in ["incident_date", "discovery_date", "disclosure_date"]:
    if col in df_inc.columns:
        df_inc = df_inc[
            df_inc[col].isna() | ((df_inc[col] >= DATE_MIN) & (df_inc[col] <= today))
        ]
print(f"Linhas removidas por data fora do range: {before - len(df_inc)}")

# --- 8. Validação de coerência temporal ---
before = len(df_inc)
mask_ok = (
    (df_inc["incident_date"].isna() | df_inc["discovery_date"].isna() | (df_inc["incident_date"] <= df_inc["discovery_date"]))
    &
    (df_inc["discovery_date"].isna() | df_inc["disclosure_date"].isna() | (df_inc["discovery_date"] <= df_inc["disclosure_date"]))
)
df_inc = df_inc[mask_ok]
print(f"Linhas removidas por incoerência temporal: {before - len(df_inc)}")

# --- 9. Descartar metadados Bronze (sempre por último) ---
df_inc = df_inc.drop(columns=[c for c in ["ingestion_timestamp", "source_file", "quality_flag"] if c in df_inc.columns])

print(f"\nincidents_master após limpeza: {df_inc.shape}")
print(f"Colunas: {list(df_inc.columns)}")

Deduplicação por incident_id: 0 removidas
Linhas removidas por data fora do range: 4
Linhas removidas por incoerência temporal: 0

incidents_master após limpeza: (846, 22)
Colunas: ['incident_id', 'company_revenue_usd', 'country_hq', 'industry_primary', 'employee_count', 'is_public_company', 'stock_ticker', 'incident_date', 'incident_date_estimated', 'discovery_date', 'disclosure_date', 'attack_vector_primary', 'attributed_group', 'attribution_confidence', 'data_compromised_records', 'data_type', 'systems_affected', 'downtime_hours', 'data_source_type', 'has_secondary_vector', 'data_loss_unknown', 'downtime_unknown']


## Limpeza — `financial_impact`

**Descartes:** `direct_loss_method`, `ransom_source`, `total_loss_method`, `cpi_index_used`, `notes`, `created_at`, `updated_at`  
**Nulos estruturais (NÃO imputar):** `ransom_demanded_usd`, `ransom_paid_usd` (só aplicável a ransomware ~27% dos casos), `regulatory_fine_usd` (só aplicável a incidentes que geraram multa)  
**Criar flags:** `is_ransomware` (presença de ransom), `has_regulatory_fine`, `insurance_unknown`  
**Imputar mediana + flag:** `insurance_payout_usd` (nulo ambíguo: sem seguro vs não reportado)

In [35]:
## Célula 5 — Limpeza: financial_impact

df_fin = raw["financial_impact"].copy()

# --- 1. Deduplicação ---
before = len(df_fin)
df_fin = df_fin.drop_duplicates(subset="incident_id", keep="first")
print(f"Deduplicação por incident_id: {before - len(df_fin)} removidas")

# --- 2. Criar flags ANTES de descartar colunas ---
# is_ransomware: ransom_demanded_usd presente indica ataque de ransomware
df_fin["is_ransomware"]      = df_fin["ransom_demanded_usd"].notna().astype(int)
# has_regulatory_fine: regulatory_fine_usd presente indica multa regulatória
df_fin["has_regulatory_fine"] = df_fin["regulatory_fine_usd"].notna().astype(int)
# insurance_unknown: insurance_payout_usd nulo (ambíguo: sem seguro ou não reportado)
df_fin["insurance_unknown"]  = df_fin["insurance_payout_usd"].isna().astype(int)

# --- 3. Colunas a descartar ---
# Metadados metodológicos: não são atributos do incidente, só descrevem como foi calculado
# ransom_source: metadado da fonte de informação sobre o ransom
# notes: texto livre; 68% nulos
# created_at / updated_at: metadado de sistema; valor idêntico para todos os registros
DISCARD_FIN = [
    "direct_loss_method",   # metadado metodológico do cálculo
    "ransom_source",        # metadado de fonte da informação de ransom
    "total_loss_method",    # metadado metodológico do cálculo
    "cpi_index_used",       # metadado do índice inflacionário utilizado
    "notes",                # texto livre; 68% nulos
    "created_at",           # metadado de sistema; zero variância
    "updated_at",           # idem
]
df_fin = df_fin.drop(columns=[c for c in DISCARD_FIN if c in df_fin.columns])

# --- 4. Nulos estruturais: NÃO imputar ---
# ransom_demanded_usd, ransom_paid_usd: nulo = "não é ataque de ransomware"
# regulatory_fine_usd: nulo = "não houve multa regulatória"
# Manter como None — a ausência é informação

# --- 5. Imputar insurance_payout_usd com mediana ---
median_ins = df_fin["insurance_payout_usd"].median()
df_fin["insurance_payout_usd"] = df_fin["insurance_payout_usd"].fillna(median_ins)

# --- 6. Descartar metadados Bronze (sempre por último) ---
df_fin = df_fin.drop(columns=[c for c in ["ingestion_timestamp", "source_file", "quality_flag"] if c in df_fin.columns])

print(f"\nfinancial_impact após limpeza: {df_fin.shape}")
print(f"Colunas: {list(df_fin.columns)}")
print(f"\nDistribuição is_ransomware:      {df_fin['is_ransomware'].value_counts().to_dict()}")
print(f"Distribuição has_regulatory_fine: {df_fin['has_regulatory_fine'].value_counts().to_dict()}")

Deduplicação por incident_id: 0 removidas

financial_impact após limpeza: (778, 15)
Colunas: ['incident_id', 'direct_loss_usd', 'ransom_demanded_usd', 'ransom_paid_usd', 'recovery_cost_usd', 'legal_fees_usd', 'regulatory_fine_usd', 'insurance_payout_usd', 'total_loss_usd', 'total_loss_lower_bound', 'total_loss_upper_bound', 'inflation_adjusted_usd', 'is_ransomware', 'has_regulatory_fine', 'insurance_unknown']

Distribuição is_ransomware:      {0: 572, 1: 206}
Distribuição has_regulatory_fine: {0: 646, 1: 132}


## Limpeza — `market_impact`

**Descartes:** `notes`, `created_at`, `updated_at`  
**Imputar mediana:** `days_to_price_recovery` (10% nulos — podem corresponder a ações que não se recuperaram no período de análise; mediana é conservadora)  
**Nota sobre perspectiva temporal:** colunas como `price_1d_after`, `abnormal_return_*`, `car_*`, `post_incident_volatility_30d` são medidas *após* o evento. Para análise retroativa são legítimas. Para predição em tempo real seriam leakage — documentado em `docs/silver_decisions.md`.

In [36]:
## Célula 6 — Limpeza: market_impact

df_mkt = raw["market_impact"].copy()

# --- 1. Deduplicação ---
before = len(df_mkt)
df_mkt = df_mkt.drop_duplicates(subset="incident_id", keep="first")
print(f"Deduplicação por incident_id: {before - len(df_mkt)} removidas")

# --- 2. Colunas a descartar ---
# notes: texto livre; 74% nulos
# created_at / updated_at: metadado de sistema; valor idêntico para todos os registros
DISCARD_MKT = [
    "notes",        # texto livre; 74% nulos
    "created_at",   # metadado de sistema; zero variância
    "updated_at",   # idem
]
df_mkt = df_mkt.drop(columns=[c for c in DISCARD_MKT if c in df_mkt.columns])

# --- 3. Padronização da categoria sector_index ---
if "sector_index" in df_mkt.columns:
    df_mkt["sector_index"] = df_mkt["sector_index"].str.strip()
    df_mkt["sector_index"] = df_mkt["sector_index"].where(df_mkt["sector_index"] != "", other=None)

# --- 4. Imputar days_to_price_recovery com mediana ---
# Nulo pode indicar recuperação não concluída no período de análise (caso mais severo)
# Imputar mediana é conservador — documentar a limitação
median_dpr = df_mkt["days_to_price_recovery"].median()
df_mkt["days_to_price_recovery"] = df_mkt["days_to_price_recovery"].fillna(median_dpr)
print(f"days_to_price_recovery: mediana={median_dpr:.1f} dias usada para imputação")

# --- 5. Descartar metadados Bronze (sempre por último) ---
df_mkt = df_mkt.drop(columns=[c for c in ["ingestion_timestamp", "source_file", "quality_flag"] if c in df_mkt.columns])

print(f"\nmarket_impact após limpeza: {df_mkt.shape}")
print(f"Colunas: {list(df_mkt.columns)}")

Deduplicação por incident_id: 0 removidas
days_to_price_recovery: mediana=57.0 dias usada para imputação

market_impact após limpeza: (358, 28)
Colunas: ['incident_id', 'stock_ticker', 'price_7d_before', 'price_disclosure_day', 'price_1d_after', 'price_7d_after', 'price_30d_after', 'volume_avg_30d_baseline', 'volume_disclosure_day', 'sector_index', 'sector_return_same_period', 'abnormal_return_1d', 'abnormal_return_7d', 'abnormal_return_30d', 'car_neg1_to_pos1', 'car_0_to_7', 'car_0_to_30', 'car_0_to_90', 't_statistic_1d', 'p_value_1d', 't_statistic_30d', 'p_value_30d', 'earnings_announcement_within_7d', 'market_cap_at_disclosure', 'volume_ratio_disclosure', 'pre_incident_volatility_30d', 'post_incident_volatility_30d', 'days_to_price_recovery']


In [37]:
## Célula 7 — Inspeção de `systems_affected` (incidents_master)

# systems_affected: 0% nulos, mas 632 valores únicos em 850 linhas
# Precisamos decidir: texto livre (descartar) ou categoria reutilizável (manter)?
print("Análise de systems_affected:")
print(f"  Valores únicos: {df_inc['systems_affected'].nunique()}")
print(f"  Total linhas  : {len(df_inc)}")
print(f"\n  Primeiros 30 valores únicos:")
for v in sorted(df_inc["systems_affected"].dropna().unique())[:30]:
    print(f"    {v}")

Análise de systems_affected:
  Valores únicos: 629
  Total linhas  : 846

  Primeiros 30 valores únicos:
    active_directory
    active_directory,backup_systems,sso,crm
    active_directory,backup_systems,sso,erp
    active_directory,cloud_infrastructure
    active_directory,cloud_infrastructure,backup_systems
    active_directory,cloud_infrastructure,crm,sso
    active_directory,cloud_infrastructure,hr_systems,erp,vpn
    active_directory,cloud_infrastructure,manufacturing_ot,email,file_servers
    active_directory,cloud_infrastructure,payment_systems
    active_directory,crm,erp
    active_directory,crm,vpn
    active_directory,customer_database,vpn
    active_directory,email,crm,file_servers
    active_directory,email,file_servers,sso,backup_systems
    active_directory,erp
    active_directory,erp,file_servers,backup_systems,email
    active_directory,file_servers
    active_directory,file_servers,customer_database
    active_directory,file_servers,manufacturing_ot,sso,cloud_infra

In [38]:
## Célula 7b — Decisão sobre systems_affected + descarte se texto livre

# Com base na inspeção acima: se cardinalidade > 50% das linhas, é texto livre → descartar
ratio = df_inc["systems_affected"].nunique() / len(df_inc)
print(f"Cardinalidade relativa de systems_affected: {ratio:.1%}")

if ratio > 0.5:
    print("→ Alta cardinalidade (>50%): texto livre ou identificador único. Descartando.")
    df_inc = df_inc.drop(columns=["systems_affected"])
else:
    print("→ Baixa cardinalidade: tratar como categoria — manter.")
    df_inc["systems_affected"] = df_inc["systems_affected"].str.strip().str.lower()
    df_inc["systems_affected"] = df_inc["systems_affected"].where(df_inc["systems_affected"] != "", other=None)

print(f"incidents_master após decisão de systems_affected: {df_inc.shape}")

Cardinalidade relativa de systems_affected: 74.3%
→ Alta cardinalidade (>50%): texto livre ou identificador único. Descartando.
incidents_master após decisão de systems_affected: (846, 21)


In [39]:
## Célula 8 — Feature engineering (incidents_master)

# Features temporais derivadas de incident_date
df_inc["incident_year"]  = df_inc["incident_date"].dt.year
df_inc["incident_month"] = df_inc["incident_date"].dt.month
df_inc["incident_day"]   = df_inc["incident_date"].dt.day

# Intervalos entre eventos (dias)
df_inc["days_to_discovery"]  = (df_inc["discovery_date"]  - df_inc["incident_date"]).dt.days
df_inc["days_to_disclosure"] = (df_inc["disclosure_date"] - df_inc["incident_date"]).dt.days

# Flags binárias de impacto
# IMPORTANTE: excluir linhas imputadas (mediana) — só considerar valores REAIS reportados.
# Sem isso, toda linha imputada com mediana > 0 conta como "houve impacto", inflando o label.
df_inc["has_data_loss"] = (
    (df_inc["data_compromised_records"] > 0) & (df_inc["data_loss_unknown"] == 0)
).astype(int)
df_inc["has_downtime"] = (
    (df_inc["downtime_hours"] > 0) & (df_inc["downtime_unknown"] == 0)
).astype(int)

# Descartar disclosure_date raw (usada apenas para derivar days_to_disclosure)
# Manter discovery_date — usada para feature days_to_discovery e é informação pré-conclusão
df_inc = df_inc.drop(columns=["disclosure_date"])

print("Features derivadas criadas:")
print("  incident_year, incident_month, incident_day")
print("  days_to_discovery, days_to_disclosure")
print("  has_data_loss, has_downtime (excluem valores imputados)")
print("  disclosure_date (raw) descartada após derivação")
print(f"\nShape atual: {df_inc.shape}")
print(f"has_data_loss: {df_inc['has_data_loss'].value_counts().sort_index().to_dict()}")
print(f"has_downtime:  {df_inc['has_downtime'].value_counts().sort_index().to_dict()}")

Features derivadas criadas:
  incident_year, incident_month, incident_day
  days_to_discovery, days_to_disclosure
  has_data_loss, has_downtime (excluem valores imputados)
  disclosure_date (raw) descartada após derivação

Shape atual: (846, 27)
has_data_loss: {0: 247, 1: 599}
has_downtime:  {0: 428, 1: 418}


In [40]:
## Célula 9 — Criar label para ML (incidents_master)

# Label: label_severe_incident
# Definição: 1 se houve perda de dados (has_data_loss) OU indisponibilidade (has_downtime)
# Justificativa: representa severidade operacional tangível e mensurável — dados comprometidos
#   e/ou sistemas fora do ar são os impactos mais relevantes para organizações.
#   - Não depende de dados financeiros (em dataset separado — evita dependência de join prematuro)
#   - Não usa disclosure_date (removida por risco de leakage temporal)
#   - Não usa quality_score / quality_grade (leakage interno)
#   - Baseado apenas em atributos do incidente observáveis na fonte primária

df_inc["label_severe_incident"] = (
    (df_inc["has_data_loss"] == 1) | (df_inc["has_downtime"] == 1)
).astype(int)

print("Label 'label_severe_incident' criado.")
print("\nDistribuição:")
vc = df_inc["label_severe_incident"].value_counts()
for val, cnt in vc.sort_index().items():
    pct = cnt / len(df_inc) * 100
    label_str = "Severo (1)" if val == 1 else "Não severo (0)"
    print(f"  {label_str}: {cnt} ({pct:.1f}%)")

balance = vc.min() / vc.max()
print(f"\nRatio de balanceamento (min/max): {balance:.2f}")
if balance < 0.3:
    print("⚠️  Classes desbalanceadas — considerar SMOTE, class_weight ou undersampling na Gold.")

Label 'label_severe_incident' criado.

Distribuição:
  Não severo (0): 118 (13.9%)
  Severo (1): 728 (86.1%)

Ratio de balanceamento (min/max): 0.16
⚠️  Classes desbalanceadas — considerar SMOTE, class_weight ou undersampling na Gold.


In [41]:
## Célula 10 — Checklist anti-leakage final

LEAKAGE_COLS = {
    "quality_score":         "incidents_master — score interno calculado após análise do incidente",
    "quality_grade":         "incidents_master — derivado do quality_score",
    "confidence_tier":       "incidents_master — metadado de curadoria interna pós-ingestão",
    "review_flag":           "incidents_master — anotação humana posterior ao incidente",
    "disclosure_date":       "incidents_master — usada apenas para days_to_disclosure; raw descartada",
    "data_source_primary":   "incidents_master — identificador de fonte, não atributo do incidente",
    "data_source_secondary": "incidents_master — idem + 55% nulos",
    "direct_loss_method":    "financial_impact — metadado metodológico do cálculo",
    "total_loss_method":     "financial_impact — metadado metodológico",
    "cpi_index_used":        "financial_impact — metadado do ajuste inflacionário",
    "ransom_source":         "financial_impact — metadado de fonte da informação de ransom",
    "created_at":            "todos — metadado de sistema; zero variância",
    "updated_at":            "todos — metadado de sistema; zero variância",
    "ingestion_timestamp":   "todos — metadado Bronze de ingestão",
    "source_file":           "todos — metadado Bronze de ingestão",
    "quality_flag":          "todos — metadado Bronze de validação",
}

print("CHECKLIST ANTI-LEAKAGE")
print("=" * 70)
silver_dfs = {
    "incidents_master":  df_inc,
    "financial_impact":  df_fin,
    "market_impact":     df_mkt,
}
all_ok = True
for col, descricao in LEAKAGE_COLS.items():
    found_in = [name for name, df in silver_dfs.items() if col in df.columns]
    status   = "✅ OK" if not found_in else f"❌ PRESENTE em {found_in}"
    if found_in:
        all_ok = False
    print(f"  {col:<30} {status}")
    print(f"     {descricao}")

print()
if all_ok:
    print("✅ Nenhuma coluna de leakage presente nos datasets Silver.")
else:
    print("❌ Colunas de leakage encontradas — revisar antes de salvar.")

CHECKLIST ANTI-LEAKAGE
  quality_score                  ✅ OK
     incidents_master — score interno calculado após análise do incidente
  quality_grade                  ✅ OK
     incidents_master — derivado do quality_score
  confidence_tier                ✅ OK
     incidents_master — metadado de curadoria interna pós-ingestão
  review_flag                    ✅ OK
     incidents_master — anotação humana posterior ao incidente
  disclosure_date                ✅ OK
     incidents_master — usada apenas para days_to_disclosure; raw descartada
  data_source_primary            ✅ OK
     incidents_master — identificador de fonte, não atributo do incidente
  data_source_secondary          ✅ OK
     incidents_master — idem + 55% nulos
  direct_loss_method             ✅ OK
     financial_impact — metadado metodológico do cálculo
  total_loss_method              ✅ OK
     financial_impact — metadado metodológico
  cpi_index_used                 ✅ OK
     financial_impact — metadado do ajuste infla

In [42]:
## Célula 11 — Validação Silver

print("VALIDAÇÃO FINAL DOS DATASETS SILVER")
print("=" * 70)

for name, df in [("incidents_master", df_inc), ("financial_impact", df_fin), ("market_impact", df_mkt)]:
    print(f"\n--- {name} ---")
    print(f"  Shape: {df.shape}")

    # Duplicatas
    dup = df.duplicated(keep=False).sum()
    dup_id = df.duplicated(subset=["incident_id"], keep=False).sum() if "incident_id" in df.columns else "n/a"
    print(f"  Duplicatas exatas: {dup} | Por incident_id: {dup_id}")

    # Nulos remanescentes
    null_counts = df.isnull().sum()
    null_counts = null_counts[null_counts > 0]
    if not null_counts.empty:
        print(f"  Nulos remanescentes ({len(null_counts)} colunas):")
        for col, cnt in null_counts.sort_values(ascending=False).items():
            print(f"    {col:<40} {cnt:>5} ({cnt/len(df)*100:.1f}%)")
    else:
        print("  ✅ Sem nulos.")

    # Schema
    print(f"  Tipos:")
    for col, dtype in df.dtypes.items():
        print(f"    {col:<40} {str(dtype)}")

# Distribuição do label
if "label_severe_incident" in df_inc.columns:
    print(f"\n--- Label: label_severe_incident ---")
    vc = df_inc["label_severe_incident"].value_counts().sort_index()
    for val, cnt in vc.items():
        print(f"  {val}: {cnt} ({cnt/len(df_inc)*100:.1f}%)")

VALIDAÇÃO FINAL DOS DATASETS SILVER

--- incidents_master ---
  Shape: (846, 28)
  Duplicatas exatas: 0 | Por incident_id: 0
  Nulos remanescentes (1 colunas):
    stock_ticker                               438 (51.8%)
  Tipos:
    incident_id                              str
    company_revenue_usd                      float64
    country_hq                               str
    industry_primary                         str
    employee_count                           int64
    is_public_company                        bool
    stock_ticker                             str
    incident_date                            datetime64[us]
    incident_date_estimated                  bool
    discovery_date                           datetime64[us]
    attack_vector_primary                    str
    attributed_group                         str
    attribution_confidence                   str
    data_compromised_records                 float64
    data_type                                str
   

In [43]:
## Célula 12 — Documentação de decisões

ts = datetime.now().strftime("%Y-%m-%d %H:%M:%S")

decisions_md = f"""# Decisões — Etapa 3 (Camada Silver)

**Gerado em:** {ts}

## Princípios aplicados

- Silver processa cada dataset de forma independente — joins entre fontes pertencem à Gold.
- A `quality_flag` da Etapa 2 foi usada para orientar decisões por linha e descartada ao final.
- Nulos foram interpretados pelo significado de negócio, não apenas pelo percentual técnico.
- Todas as colunas foram analisadas — silêncio do relatório Bronze não significa aprovação para ML.

---

## incidents_master

### Descartadas
| Coluna | Motivo |
|--------|--------|
| company_name | Identificador de alta cardinalidade (750 únicos); não é feature preditiva |
| industry_secondary | 82% nulos estruturais; vetor primário já presente |
| attack_vector_secondary | 75% nulos estruturais; substituído por flag has_secondary_vector |
| attack_chain | Texto narrativo livre; não adequado para ML clássico |
| data_source_primary | Identificador de fonte jornalística; não é atributo do incidente |
| data_source_secondary | Idem + 55% nulos |
| quality_score | **LEAKAGE** — score interno calculado após análise do incidente |
| quality_grade | **LEAKAGE** — derivado de quality_score |
| confidence_tier | **LEAKAGE** — metadado de curadoria interna pós-ingestão |
| review_flag | **LEAKAGE** — anotação humana posterior ao incidente |
| notes | Texto livre; 75% nulos |
| created_at | Metadado de sistema; zero variância (valor idêntico para todos) |
| updated_at | Idem |
| disclosure_date (raw) | Usada apenas para derivar days_to_disclosure; raw descartada por risco de leakage temporal |

### Tratamento de nulos
| Coluna | Tipo de Nulo | Tratamento |
|--------|-------------|------------|
| stock_ticker | Não aplicável (empresa privada sem ticker) | Mantido como None |
| attributed_group | Não atribuído / desconhecido | Imputado "unknown" |
| attribution_confidence | Condicionado a attributed_group | Imputado "unknown" |
| data_type | Condicionado a data_compromised_records | "none" se sem exfiltração; "unknown" nos demais |
| data_compromised_records | Ambíguo (sem exfiltração OU não reportado) | Mediana + flag data_loss_unknown |
| downtime_hours | Ambíguo (sem downtime OU não reportado) | Mediana + flag downtime_unknown (imputar 0 seria incorreto) |

### Features criadas
| Feature | Definição |
|---------|-----------|
| has_secondary_vector | 1 se o incidente tinha vetor de ataque secundário |
| incident_year/month/day | Componentes temporais de incident_date |
| days_to_discovery | discovery_date - incident_date (dias) |
| days_to_disclosure | disclosure_date - incident_date (dias); disclosure_date raw descartada |
| has_data_loss | 1 se data_compromised_records > 0 |
| has_downtime | 1 se downtime_hours > 0 |
| data_loss_unknown | 1 se data_compromised_records era nulo na Bronze |
| downtime_unknown | 1 se downtime_hours era nulo na Bronze |
| label_severe_incident | **Label de ML** — ver seção abaixo |

### Label: label_severe_incident
**Definição:** 1 se `has_data_loss == 1` OU `has_downtime == 1`.  
**Justificativa:** representa severidade operacional tangível. Dados comprometidos e/ou sistemas
fora do ar são os impactos mais relevantes e mensuráveis para organizações.  
Não usa colunas financeiras (em dataset separado — evita dependência de join prematuro).  
Não usa disclosure_date nem quality_score (leakage).  
Baseado apenas em atributos do incidente observáveis na fonte primária.

---

## financial_impact

### Descartadas
| Coluna | Motivo |
|--------|--------|
| direct_loss_method | Metadado metodológico do cálculo; não é atributo do incidente |
| ransom_source | Metadado de fonte da informação de ransom |
| total_loss_method | Metadado metodológico |
| cpi_index_used | Metadado do índice inflacionário utilizado |
| notes | Texto livre; 68% nulos |
| created_at | Metadado de sistema; zero variância |
| updated_at | Idem |

### Tratamento de nulos
| Coluna | Tipo de Nulo | Tratamento |
|--------|-------------|------------|
| ransom_demanded_usd | Não aplicável (apenas ransomware ~27% dos casos) | Mantido como None — ausência é informação |
| ransom_paid_usd | Condicionado a ransom + decisão de pagamento | Mantido como None |
| regulatory_fine_usd | Não aplicável (apenas incidentes que geraram multa) | Mantido como None |
| insurance_payout_usd | Ambíguo (sem seguro OU não reportado) | Mediana + flag insurance_unknown |

### Flags criadas
| Flag | Definição |
|------|-----------|
| is_ransomware | 1 se ransom_demanded_usd não era nulo na Bronze |
| has_regulatory_fine | 1 se regulatory_fine_usd não era nulo na Bronze |
| insurance_unknown | 1 se insurance_payout_usd era nulo na Bronze |

---

## market_impact

### Descartadas
| Coluna | Motivo |
|--------|--------|
| notes | Texto livre; 74% nulos |
| created_at | Metadado de sistema; zero variância |
| updated_at | Idem |

### Tratamento de nulos
| Coluna | Tipo de Nulo | Tratamento |
|--------|-------------|------------|
| days_to_price_recovery | Possível recuperação não concluída no período | Mediana (conservador) |

### Nota sobre perspectiva temporal
Colunas como `price_1d_after`, `abnormal_return_*`, `car_*`, `post_incident_volatility_30d`
são medidas **após** o evento de divulgação. Para **análise retroativa** são features legítimas.
Para **predição em tempo real** seriam data leakage severo — não observáveis antes do evento.
O uso adequado depende do objetivo do modelo a ser construído na Gold.
"""

anti_leakage_md = f"""# Checklist Anti-Leakage — Silver

**Gerado em:** {ts}

| Coluna | Dataset(s) | Risco | Decisão |
|--------|-----------|-------|---------|
| quality_score | incidents_master | Score interno calculado após o incidente | Removida |
| quality_grade | incidents_master | Derivado de quality_score | Removida |
| confidence_tier | incidents_master | Metadado de curadoria interna | Removida |
| review_flag | incidents_master | Anotação humana pós-incidente | Removida |
| disclosure_date (raw) | incidents_master | Posterior ao incidente; revela severidade | Removida após derivação de days_to_disclosure |
| data_source_primary | incidents_master | Identificador de fonte, não atributo do incidente | Removida |
| data_source_secondary | incidents_master | Idem | Removida |
| direct_loss_method | financial_impact | Metadado metodológico | Removida |
| total_loss_method | financial_impact | Metadado metodológico | Removida |
| cpi_index_used | financial_impact | Metadado do ajuste inflacionário | Removida |
| ransom_source | financial_impact | Metadado de fonte | Removida |
| created_at | todos | Zero variância; metadado de sistema | Removida |
| updated_at | todos | Zero variância; metadado de sistema | Removida |
| ingestion_timestamp | todos | Metadado de pipeline Bronze | Removida |
| source_file | todos | Metadado de pipeline Bronze | Removida |
| quality_flag | todos | Metadado de validação Bronze | Usada durante limpeza; removida ao final |

## Colunas com risco condicional (mantidas com documentação)

| Coluna | Dataset | Situação |
|--------|---------|----------|
| price_1d_after, abnormal_return_*, car_*, post_incident_volatility_30d | market_impact | Legítimas para análise retroativa; leakage em predição tempo real |
| days_to_disclosure | incidents_master | Feature derivada; disclosure_date raw removida |
| ransom_demanded_usd, ransom_paid_usd | financial_impact | Nulos estruturais mantidos como None; não imputados |
"""

(DOCS_PATH / "silver_decisions.md").write_text(decisions_md, encoding="utf-8")
(DOCS_PATH / "anti_leakage_checklist.md").write_text(anti_leakage_md, encoding="utf-8")

print("Documentos gerados:")
print(f"  {DOCS_PATH / 'silver_decisions.md'}")
print(f"  {DOCS_PATH / 'anti_leakage_checklist.md'}")

Documentos gerados:
  C:\Users\dti-\Pessoal\CC5\ciencia-de-dados\cybersecurity-breach-data-project\docs\silver_decisions.md
  C:\Users\dti-\Pessoal\CC5\ciencia-de-dados\cybersecurity-breach-data-project\docs\anti_leakage_checklist.md


In [44]:
## Célula 13 — Salvar Silver em Parquet

silver_datasets = {
    "incidents_master_silver":  df_inc,
    "financial_impact_silver":  df_fin,
    "market_impact_silver":     df_mkt,
}

for name, df in silver_datasets.items():
    out_path = SILVER_PATH / f"{name}.parquet"
    df.to_parquet(out_path, index=False)
    size_kb = out_path.stat().st_size / 1024
    print(f"✅ {name}.parquet — {len(df)} linhas × {len(df.columns)} colunas — {size_kb:.1f} KB")
    print(f"   Salvo em: {out_path}")

print("\nCamada Silver gerada com sucesso.")
print(f"Arquivos em: {SILVER_PATH}")

✅ incidents_master_silver.parquet — 846 linhas × 28 colunas — 64.2 KB


   Salvo em: C:\Users\dti-\Pessoal\CC5\ciencia-de-dados\cybersecurity-breach-data-project\data\silver\incidents_master_silver.parquet
✅ financial_impact_silver.parquet — 778 linhas × 15 colunas — 66.8 KB
   Salvo em: C:\Users\dti-\Pessoal\CC5\ciencia-de-dados\cybersecurity-breach-data-project\data\silver\financial_impact_silver.parquet
✅ market_impact_silver.parquet — 358 linhas × 28 colunas — 86.0 KB
   Salvo em: C:\Users\dti-\Pessoal\CC5\ciencia-de-dados\cybersecurity-breach-data-project\data\silver\market_impact_silver.parquet

Camada Silver gerada com sucesso.
Arquivos em: C:\Users\dti-\Pessoal\CC5\ciencia-de-dados\cybersecurity-breach-data-project\data\silver
